# tt-mlir #9295 빌드 + lit test (OPMODEL=ON)

**할 일은 딱 둘:**
1. 우상단 런타임(Runtime) → "런타임 유형 변경" → **CPU, 고RAM(High-RAM)** 선택 (GPU 불필요)
2. 메뉴 Runtime → **Run all** 클릭

이후로는 셀 다 자동 실행됨. **이번 빌드는 이전(OPMODEL=OFF)보다 훨씬 오래 걸림** —
`ttnn-greedy-memory-layout-propagation` pass 자체가 컴파일타임에 opmodel 지원을
요구해서(`#ifndef TTMLIR_ENABLE_OPMODEL` → abort), OPMODEL=OFF 빌드로는 이 테스트를
절대 통과시킬 수 없다는 게 확인됨. OPMODEL=ON은 tt-metal 서브프로젝트를 함께 빌드하므로
toolchain+본체 합쳐 체감 2시간 이상 걸릴 수 있음(하드웨어는 불필요 — NO_DISPATCH 모드로 빌드).

In [ ]:
!nproc
!free -h


In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq clang ninja-build cmake git python3.12-venv libgtest-dev libgmock-dev


In [ ]:
%cd /content
!rm -rf /content/tt-mlir
!git clone --branch fix-9295-multi-result-dram-fallback https://github.com/alexxony/tt-mlir.git /content/tt-mlir
%cd /content/tt-mlir
!git log --oneline -3


In [ ]:
import os
os.environ["TTMLIR_TOOLCHAIN_DIR"] = "/opt/ttmlir-toolchain/"
!mkdir -p /opt/ttmlir-toolchain
!chown -R $(whoami) /opt/ttmlir-toolchain


## Toolchain 빌드 (LLVM 등) — 가장 오래 걸림, 20~40분

In [ ]:
%cd /content/tt-mlir
!cmake -B env/build env -DCMAKE_C_COMPILER=clang -DCMAKE_CXX_COMPILER=clang++
!cmake --build env/build


## tt-mlir 본체 빌드 (OPMODEL=ON — tt-metal 서브프로젝트 포함, 훨씬 오래 걸림)

런타임(RUNTIME)은 여전히 꺼둠 — lit test에 필요한 건 OPMODEL뿐. Python 바인딩도 꺼서
빌드 범위를 최소화.

In [ ]:
%cd /content/tt-mlir
!bash -c "source env/activate && cmake -G Ninja -B build -DCMAKE_BUILD_TYPE=Release -DTTMLIR_ENABLE_RUNTIME=OFF -DTTMLIR_ENABLE_OPMODEL=ON -DTTMLIR_ENABLE_BINDINGS_PYTHON=OFF -DCMAKE_BUILD_PARALLEL_LEVEL=$(nproc)"
!bash -c "source env/activate && cmake --build build -- -j$(nproc)" 2>&1 | tee /content/main_build.log | tail -150


## 본체 빌드 결과 확인

위 셀이 실패했으면 여기서 원인 로그를 바로 봄(tt-metal 관련 새 의존성 에러 등).
성공(exit 0)이면 다음 lit test 셀로.

In [ ]:
import subprocess
r = subprocess.run(["tail", "-n", "60", "/content/main_build.log"], capture_output=True, text=True)
print(r.stdout)
# 빌드가 실제로 성공했는지 바이너리 존재로 확인
import os
print("ttmlir-opt exists:", os.path.exists("/content/tt-mlir/build/bin/ttmlir-opt"))


## lit test 실행 (신규 테스트만 우선 단독 실행)

In [ ]:
%cd /content/tt-mlir
import subprocess
TEST = "test/ttmlir/Dialect/TTNN/optimizer/op_layout_fallbacks/multi_result_dram_fallback.mlir"
r = subprocess.run(["bash", "-c", f"source env/activate && llvm-lit {TEST} -v"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)
with open("/content/multi_result_test.log", "w") as f:
    f.write(r.stdout + r.stderr)


## 진단: FileCheck 실제 diff 확보 (신규 테스트가 여전히 FAIL이면)

`ttmlir-opt`를 직접 돌려서 실제 산출 IR과 FileCheck가 정확히 뭘 기대하고 뭘 받았는지
텍스트로 확보한다. `lit -v`의 diff가 잘려 보일 때 대비용.

In [ ]:
%cd /content/tt-mlir
TEST = "test/ttmlir/Dialect/TTNN/optimizer/op_layout_fallbacks/multi_result_dram_fallback.mlir"

import subprocess
r1 = subprocess.run(
    ["bash", "-c",
     f"source env/activate && ttmlir-opt --ttcore-register-device "
     f"--ttcore-mark-functions-as-forward --ttnn-greedy-memory-layout-propagation "
     f"{TEST} --mlir-print-local-scope -o /content/actual_out.mlir"],
    capture_output=True, text=True,
)
print("===== ttmlir-opt exit code:", r1.returncode, "=====")
print("----- stderr -----")
print(r1.stderr)

print("===== /content/actual_out.mlir (실제 산출 IR) =====")
try:
    with open("/content/actual_out.mlir") as f:
        print(f.read())
except FileNotFoundError:
    print("(파일 생성 안 됨 -- ttmlir-opt가 크래시했을 가능성, 위 stderr 확인)")

r2 = subprocess.run(
    ["bash", "-c",
     f"source env/activate && FileCheck {TEST} "
     f"--input-file /content/actual_out.mlir --dump-input=always"],
    capture_output=True, text=True,
)
print("===== FileCheck exit code:", r2.returncode, "(0=PASS) =====")
print(r2.stdout)
print(r2.stderr)

with open("/content/filecheck_diag.log", "w") as f:
    f.write("EXIT_CODE=" + str(r2.returncode) + "\n")
    f.write(r2.stdout)
    f.write(r2.stderr)


## 전체 check-ttmlir (선택 -- 신규 테스트 PASS 확인 후 회귀 여부 보고 싶으면 실행)

OPMODEL=ON이면 전체 스위트 실행 시간도 늘어남(hardware mock 검증 경로가 늘어난 테스트들이
이제 SKIP 아니라 실제 실행됨). 급하지 않으면 건너뛰어도 됨 -- PR은 신규 테스트 결과만으로도
충분, 전체 회귀는 CI 몫.

In [ ]:
%cd /content/tt-mlir
!bash -c "source env/activate && cmake --build build -- check-ttmlir" 2>&1 | tee /content/lit_test_full.log | tail -150


## 결과 요약

In [ ]:
import re

print("=" * 60)
print("신규 테스트(multi_result_dram_fallback.mlir) 결과")
print("=" * 60)
with open("/content/multi_result_test.log") as f:
    print(f.read())

print("=" * 60)
print("FileCheck 진단 로그 (있으면)")
print("=" * 60)
try:
    with open("/content/filecheck_diag.log") as f:
        print(f.read())
except FileNotFoundError:
    print("(신규 테스트가 이미 PASS라 진단 셀 스킵됐거나 실행 안 됨)")

print("=" * 60)
print("전체 check-ttmlir PASS/FAIL 요약 (실행했으면)")
print("=" * 60)
try:
    with open("/content/lit_test_full.log") as f:
        text = f.read()
    summary_lines = [l for l in text.splitlines() if re.search(
        r"Testing Time|Passed|Failed|Total Discovered Tests|Expected|Unsupported|^FAIL:|^PASS:", l)]
    print(chr(10).join(summary_lines[-40:]) if summary_lines else chr(10).join(text.splitlines()[-40:]))
except FileNotFoundError:
    print("(전체 스위트 셀 스킵됨)")


## 산출물 다운로드 (선택)

결과를 로컬(WSL)로 가져가고 싶으면 아래 셀 실행 후, `Files` 사이드바에서 `/content/backup_bundle` 우클릭 → 다운로드.
또는 그냥 위 결과 요약 셀 출력만 스크린샷/복사해서 알려줘도 됨.

In [ ]:
import shutil, os
os.makedirs("/content/backup_bundle", exist_ok=True)
for f in ["main_build.log", "multi_result_test.log", "filecheck_diag.log", "actual_out.mlir", "lit_test_full.log"]:
    src = f"/content/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"/content/backup_bundle/{f}")
if os.path.exists("/content/tt-mlir/build/bin/ttmlir-opt"):
    shutil.copy("/content/tt-mlir/build/bin/ttmlir-opt", "/content/backup_bundle/ttmlir-opt")
print("done:", os.listdir("/content/backup_bundle"))


## ⚠️ 실행 끝나면 GitHub에 저장 (결과 유실 방지)

Colab 메뉴 **파일(File) → GitHub에 사본 저장(Save a copy in GitHub)** 클릭 →
같은 브랜치(`fix-9295-multi-result-dram-fallback`)에 커밋. **이 단계를 건너뛰면
위 결과가 로컬(claude)에서 안 보임** — 이전 세션에서 실제로 한 번 유실됐던 지점.